# **AI Advanced Ahmed Yousrey Course Practice**

---
# **DAY 3 — Job Application Intelligence**
---

## STEP 1 — Importing Libraries

---

In [10]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings("ignore")

---
## STEP 2 — Load Data
---

In [11]:
df = pd.read_csv("/content/job_applicant_dataset.csv")
df.head()

,Job Applicant Name,Age,Gender,Race,Ethnicity,Resume,Job Roles,Job Description,Best Match
0,Daisuke Mori,29,Male,Mongoloid/Asian,Vietnamese,"Proficient in Injury Prevention, Motivation, N...",Fitness Coach,A Fitness Coach is responsible for helping cl...,0
1,Taichi Shimizu,31,Male,Mongoloid/Asian,Filipino,"Proficient in Healthcare, Pharmacology, Medica...",Physician,"Diagnose and treat illnesses, prescribe medica...",0
2,Sarah Martin,46,Female,White/Caucasian,Dutch,"Proficient in Forecasting, Financial Modelling...",Financial Analyst,"As a Financial Analyst, you will be responsibl...",0
3,Keith Hughes,43,Male,Negroid/Black,Caribbean,"Proficient in Budgeting, Supply Chain Optimiza...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1
4,James Davis,49,Male,White/Caucasian,English,"Proficient in Logistics, Negotiation, Procurem...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1


In [12]:
df = df.drop(['Job Applicant Name', 'Race', 'Ethnicity'], axis=1)

In [13]:
df.head()

,Age,Gender,Resume,Job Roles,Job Description,Best Match
0,29,Male,"Proficient in Injury Prevention, Motivation, N...",Fitness Coach,A Fitness Coach is responsible for helping cl...,0
1,31,Male,"Proficient in Healthcare, Pharmacology, Medica...",Physician,"Diagnose and treat illnesses, prescribe medica...",0
2,46,Female,"Proficient in Forecasting, Financial Modelling...",Financial Analyst,"As a Financial Analyst, you will be responsibl...",0
3,43,Male,"Proficient in Budgeting, Supply Chain Optimiza...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1
4,49,Male,"Proficient in Logistics, Negotiation, Procurem...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1


---
## STEP 3 — EDA
---

In [14]:
df.shape

(10000, 6)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Age              10000 non-null  int64 
 1   Gender           10000 non-null  object
 2   Resume           10000 non-null  object
 3   Job Roles        10000 non-null  object
 4   Job Description  10000 non-null  object
 5   Best Match       10000 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 468.9+ KB


there is no missing values

In [16]:
df.duplicated().sum()

np.int64(0)

In [17]:
df.head()

,Age,Gender,Resume,Job Roles,Job Description,Best Match
0,29,Male,"Proficient in Injury Prevention, Motivation, N...",Fitness Coach,A Fitness Coach is responsible for helping cl...,0
1,31,Male,"Proficient in Healthcare, Pharmacology, Medica...",Physician,"Diagnose and treat illnesses, prescribe medica...",0
2,46,Female,"Proficient in Forecasting, Financial Modelling...",Financial Analyst,"As a Financial Analyst, you will be responsibl...",0
3,43,Male,"Proficient in Budgeting, Supply Chain Optimiza...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1
4,49,Male,"Proficient in Logistics, Negotiation, Procurem...",Supply Chain Manager,A Supply Chain Manager oversees the entire sup...,1


---
## STEP 4 — PREPROCESSING
---

split data

In [18]:

x = df.drop("Best Match", axis=1)
y = df["Best Match"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

fix data types

In [19]:
map={
    'Male':1,
    'Female':0
}
x_train['Gender'] = x_train['Gender'].map(map)
x_test['Gender'] = x_test['Gender'].map(map)

text preprocessing

1 - resume TF-IDF

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_resume = TfidfVectorizer()

resume_train_tfidf = tfidf_resume.fit_transform(x_train["Resume"])
resume_test_tfidf = tfidf_resume.transform(x_test["Resume"])

In [21]:
print(resume_train_tfidf.shape)
print(resume_test_tfidf.shape)

(8000, 682)
(2000, 682)


In [22]:
resume_train_tfidf = tfidf_resume.fit_transform(x_train["Resume"])

2 - Job Description

In [23]:
tfidf_job_desc = TfidfVectorizer()

job_desc_train_tfidf = tfidf_job_desc.fit_transform(
    x_train["Job Description"]
)

job_desc_test_tfidf = tfidf_job_desc.transform(
    x_test["Job Description"]
)

In [24]:
print(job_desc_train_tfidf.shape)
print(job_desc_test_tfidf.shape)

(8000, 1134)
(2000, 1134)


3 - job role

In [25]:
tfidf_job_role = TfidfVectorizer()

job_role_train_tfidf = tfidf_job_role.fit_transform(
    x_train["Job Roles"]
)

job_role_test_tfidf = tfidf_job_role.transform(
    x_test["Job Roles"]
)

In [26]:
print(job_role_train_tfidf.shape)
print(job_role_test_tfidf.shape)

(8000, 70)
(2000, 70)


compine the 3 TF-IDF matrices

In [27]:
from scipy.sparse import hstack

x_train_text = hstack([
    resume_train_tfidf,
    job_desc_train_tfidf,
    job_role_train_tfidf
])

x_test_text = hstack([
    resume_test_tfidf,
    job_desc_test_tfidf,
    job_role_test_tfidf
])

In [28]:
print(x_train_text.shape)
print(x_test_text.shape)

(8000, 1886)
(2000, 1886)


feature categorization

In [29]:
numeric_train = x_train[["Age", "Gender"]].values
numeric_test = x_test[["Age", "Gender"]].values

In [30]:
print(numeric_train.shape)
print(numeric_test.shape)

(8000, 2)
(2000, 2)


add numerical value to the TF-IDF matrix

In [31]:
from scipy.sparse import csr_matrix, hstack

x_train_final = hstack([
    x_train_text,
    csr_matrix(numeric_train)
])

x_test_final = hstack([
    x_test_text,
    csr_matrix(numeric_test)
])

In [33]:
print(x_train_final.shape)
print(x_test_final.shape)

(8000, 1888)
(2000, 1888)


---
## STEP 5 — Model Training & Evaluation
---

LogisticRegression

In [34]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(x_train_final, y_train)

LogisticRegression(max_iter=1000)

In [36]:
y_pred_lr = lr.predict(x_test_final)
y_pred_lr

array([0, 0, 0, ..., 0, 1, 1])

In [40]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.6315
              precision    recall  f1-score   support

           0       0.64      0.64      0.64      1030
           1       0.62      0.62      0.62       970

    accuracy                           0.63      2000
   macro avg       0.63      0.63      0.63      2000
weighted avg       0.63      0.63      0.63      2000



LinearSVC

In [41]:
from sklearn.svm import LinearSVC

svc = LinearSVC()

svc.fit(x_train_final, y_train)

y_pred_svc = svc.predict(x_test_final)

In [43]:
print("Accuracy:", accuracy_score(y_test, y_pred_svc))
print(classification_report(y_test, y_pred_svc))

Accuracy: 0.6305
              precision    recall  f1-score   support

           0       0.64      0.64      0.64      1030
           1       0.62      0.62      0.62       970

    accuracy                           0.63      2000
   macro avg       0.63      0.63      0.63      2000
weighted avg       0.63      0.63      0.63      2000



naive_bayes

In [44]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(x_train_final, y_train)

y_pred_nb = nb.predict(x_test_final)

In [46]:
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Accuracy: 0.549
              precision    recall  f1-score   support

           0       0.58      0.46      0.51      1030
           1       0.53      0.64      0.58       970

    accuracy                           0.55      2000
   macro avg       0.55      0.55      0.55      2000
weighted avg       0.55      0.55      0.55      2000



teet if data is inbalance

In [47]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Best Match
0    0.515
1    0.485
Name: proportion, dtype: float64
Best Match
0    0.515
1    0.485
Name: proportion, dtype: float64


the results is not satisfying

---

try to increase number of features the TF-IDF extract and repeat the same steps again

In [48]:
tfidf_resume_2 = TfidfVectorizer(
    ngram_range=(1, 2)
)

resume_train_tfidf_2 = tfidf_resume_2.fit_transform(
    x_train["Resume"]
)

resume_test_tfidf_2 = tfidf_resume_2.transform(
    x_test["Resume"]
)

print(resume_train_tfidf_2.shape)
print(resume_test_tfidf_2.shape)

(8000, 5619)
(2000, 5619)


In [49]:
tfidf_job_desc_2 = TfidfVectorizer(
    ngram_range=(1, 2)
)

job_desc_train_tfidf_2 = tfidf_job_desc_2.fit_transform(
    x_train["Job Description"]
)

job_desc_test_tfidf_2 = tfidf_job_desc_2.transform(
    x_test["Job Description"]
)

print(job_desc_train_tfidf_2.shape)
print(job_desc_test_tfidf_2.shape)

(8000, 4277)
(2000, 4277)


In [50]:
tfidf_job_role_2 = TfidfVectorizer(
    ngram_range=(1, 2)
)

job_role_train_tfidf_2 = tfidf_job_role_2.fit_transform(
    x_train["Job Roles"]
)

job_role_test_tfidf_2 = tfidf_job_role_2.transform(
    x_test["Job Roles"]
)

print(job_role_train_tfidf_2.shape)
print(job_role_test_tfidf_2.shape)

(8000, 110)
(2000, 110)


In [51]:
x_train_text_2 = hstack([
    resume_train_tfidf_2,
    job_desc_train_tfidf_2,
    job_role_train_tfidf_2
])

x_test_text_2 = hstack([
    resume_test_tfidf_2,
    job_desc_test_tfidf_2,
    job_role_test_tfidf_2
])

In [52]:
x_train_final_2 = hstack([
    x_train_text_2,
    csr_matrix(numeric_train)
])

x_test_final_2 = hstack([
    x_test_text_2,
    csr_matrix(numeric_test)
])

print(x_train_final_2.shape)
print(x_test_final_2.shape)

(8000, 10008)
(2000, 10008)


In [53]:
lr_2 = LogisticRegression(max_iter=1000)

lr_2.fit(x_train_final_2, y_train)

y_pred_lr_2 = lr_2.predict(x_test_final_2)

print("Accuracy:", accuracy_score(y_test, y_pred_lr_2))
print(classification_report(y_test, y_pred_lr_2))

Accuracy: 0.632
              precision    recall  f1-score   support

           0       0.64      0.64      0.64      1030
           1       0.62      0.62      0.62       970

    accuracy                           0.63      2000
   macro avg       0.63      0.63      0.63      2000
weighted avg       0.63      0.63      0.63      2000



there is no enough improvement

---


test and see the relation between the extracted features from **(resume)** column and **(Job Description)**

make them first have the same size so we can compare them by using **cosine_similarity**

In [58]:
tfidf_match = TfidfVectorizer()

resume_job_train = x_train["Resume"] + " " + x_train["Job Description"]
resume_job_test = x_test["Resume"] + " " + x_test["Job Description"]

tfidf_match.fit(
    resume_job_train
)

resume_train_match = tfidf_match.transform(
    x_train["Resume"]
)

job_desc_train_match = tfidf_match.transform(
    x_train["Job Description"]
)

resume_test_match = tfidf_match.transform(
    x_test["Resume"]
)

job_desc_test_match = tfidf_match.transform(
    x_test["Job Description"]
)

In [59]:
print(resume_train_match.shape)
print(job_desc_train_match.shape)

(8000, 1535)
(8000, 1535)


**cosine_similarity**

In [60]:
similarity_train = cosine_similarity(
    resume_train_match,
    job_desc_train_match
).diagonal()

similarity_test = cosine_similarity(
    resume_test_match,
    job_desc_test_match
).diagonal()

In [61]:
print(similarity_train[:10])
print(similarity_test[:10])

[0.20956676 0.10271614 0.17919331 0.27291826 0.32837393 0.33091418
 0.07491068 0.21345696 0.25244785 0.42148008]
[0.12501641 0.26135273 0.21806755 0.48089272 0.08638068 0.38684357
 0.14327327 0.05605395 0.30493485 0.15864972]


This is the cosine similarity value for each resume against its corresponding job description.

In [62]:
import pandas as pd

similarity_df = pd.DataFrame({
    "similarity": similarity_train,
    "best_match": y_train.values
})

print(similarity_df.groupby("best_match")["similarity"].mean())

best_match
0    0.227274
1    0.225134
Name: similarity, dtype: float64


test if there is inbalance in the **best_match**

In [63]:
role_similarity_df = pd.DataFrame({
    "job_role": x_train["Job Roles"].values,
    "best_match": y_train.values
})

print(role_similarity_df.groupby("best_match").size())


best_match
0    4120
1    3880
dtype: int64


see the correlation relation between the **similarity  ↔  best_match**

In [64]:
print(
    similarity_df["similarity"].corr(
        similarity_df["best_match"]
    )
)

-0.009452594590227537


Try to convert the similarity data from a standard array into a sparse matrix so that we can add it to our TF-IDF matrix using `hstack`.

In [65]:
similarity_train_sparse = csr_matrix(
    similarity_train.reshape(-1, 1)
)

similarity_test_sparse = csr_matrix(
    similarity_test.reshape(-1, 1)
)

In [66]:
x_train_with_similarity = hstack([
    x_train_final,
    similarity_train_sparse
])

x_test_with_similarity = hstack([
    x_test_final,
    similarity_test_sparse
])

print(x_train_with_similarity.shape)
print(x_test_with_similarity.shape)

(8000, 1889)
(2000, 1889)


try evaluate after the changes

In [70]:
lr_similarity = LogisticRegression(max_iter=1000)

lr_similarity.fit(
    x_train_with_similarity,
    y_train
)

y_pred_lr_similarity = lr_similarity.predict(
    x_test_with_similarity
)

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_lr_similarity))
print(classification_report(y_test, y_pred_lr_similarity))

Accuracy: 0.6315
              precision    recall  f1-score   support

           0       0.64      0.64      0.64      1030
           1       0.62      0.62      0.62       970

    accuracy                           0.63      2000
   macro avg       0.63      0.63      0.63      2000
weighted avg       0.63      0.63      0.63      2000



----
**Step 6**

**Rebuild with Pipelines**

----

In [71]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# Reload fresh splits (Gender still needs encoding — do it on raw df)
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

X = df.drop("Best Match", axis=1)
y = df["Best Match"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define which columns get which treatment
text_columns = ["Resume", "Job Description", "Job Roles"]
numeric_columns = ["Age", "Gender"]

# ColumnTransformer: apply TF-IDF to each text column, passthrough numerics
preprocessor = ColumnTransformer(transformers=[
    ("resume",   TfidfVectorizer(), "Resume"),
    ("job_desc", TfidfVectorizer(), "Job Description"),
    ("job_role", TfidfVectorizer(), "Job Roles"),
    ("numeric",  "passthrough",     numeric_columns)
])

# Build the full pipeline: preprocess → model
pipeline_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(max_iter=1000))
])

# One call does everything: fit TF-IDFs on train, transform, train model
pipeline_lr.fit(X_train, y_train)

# One call does everything: transform test, predict
y_pred = pipeline_lr.predict(X_test)

print("=== Logistic Regression (Pipeline) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

=== Logistic Regression (Pipeline) ===
Accuracy: 0.632
              precision    recall  f1-score   support

           0       0.64      0.64      0.64      1030
           1       0.62      0.62      0.62       970

    accuracy                           0.63      2000
   macro avg       0.63      0.63      0.63      2000
weighted avg       0.63      0.63      0.63      2000



----
**Step 7**

**Cross-Validation Comparison**

----

In [72]:

from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB

# Define all three pipelines
pipeline_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

pipeline_svc = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(max_iter=2000))
])

# Note: MultinomialNB requires non-negative features
# passthrough on numeric columns is fine here since Age and Gender >= 0
pipeline_nb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", MultinomialNB())
])

models = {
    "Logistic Regression": pipeline_lr,
    "Linear SVC":          pipeline_svc,
    "Naive Bayes":         pipeline_nb,
}

print("=== 5-Fold Cross-Validation Results ===\n")

cv_results = {}

for name, pipeline in models.items():
    scores = cross_val_score(
        pipeline,
        X,          # full dataset — CV does its own splitting
        y,
        cv=5,
        scoring="accuracy"
    )
    cv_results[name] = scores
    print(f"{name}")
    print(f"  Scores : {scores.round(3)}")
    print(f"  Mean   : {scores.mean():.3f}")
    print(f"  Std    : {scores.std():.3f}")
    print()

=== 5-Fold Cross-Validation Results ===

Logistic Regression
  Scores : [0.664 0.655 0.656 0.648 0.658]
  Mean   : 0.656
  Std    : 0.005

Linear SVC
  Scores : [0.654 0.652 0.652 0.64  0.655]
  Mean   : 0.651
  Std    : 0.005

Naive Bayes
  Scores : [0.54  0.52  0.522 0.55  0.524]
  Mean   : 0.531
  Std    : 0.012



----
**Step 8**

**Ensemble Models**

----

In [73]:

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier

# --- Random Forest Pipeline ---
pipeline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

# --- Gradient Boosting Pipeline ---
pipeline_gb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(n_estimators=100, random_state=42))
])

ensemble_models = {
    "Random Forest":       pipeline_rf,
    "Gradient Boosting":   pipeline_gb,
}

print("=== Ensemble Models — 5-Fold Cross-Validation ===\n")

for name, pipeline in ensemble_models.items():
    scores = cross_val_score(pipeline, X, y, cv=5, scoring="accuracy")
    cv_results[name] = scores
    print(f"{name}")
    print(f"  Scores : {scores.round(3)}")
    print(f"  Mean   : {scores.mean():.3f}")
    print(f"  Std    : {scores.std():.3f}")
    print()

# --- Full comparison table ---
print("=== Full Model Comparison ===\n")
print(f"{'Model':<25} {'Mean Accuracy':>15} {'Std':>8}")
print("-" * 50)
for name, scores in cv_results.items():
    print(f"{name:<25} {scores.mean():>15.3f} {scores.std():>8.3f}")

=== Ensemble Models — 5-Fold Cross-Validation ===

Random Forest
  Scores : [0.842 0.847 0.834 0.83  0.849]
  Mean   : 0.840
  Std    : 0.007

Gradient Boosting
  Scores : [0.866 0.874 0.868 0.874 0.886]
  Mean   : 0.874
  Std    : 0.007

=== Full Model Comparison ===

Model                       Mean Accuracy      Std
--------------------------------------------------
Logistic Regression                 0.656    0.005
Linear SVC                          0.651    0.005
Naive Bayes                         0.531    0.012
Random Forest                       0.840    0.007
Gradient Boosting                   0.874    0.007


----
**Step 9**

**Hyperparameter Tuning**

----

In [ ]:
from sklearn.model_selection import GridSearchCV



# Define the parameter grid

# Note: parameters inside a pipeline are accessed as "stepname__parameter"

param_grid = {

    "classifier__n_estimators":   [100, 200, 300],

    "classifier__max_depth":      [None, 10, 20],

    "classifier__min_samples_split": [2, 5, 10],

}



# Build the pipeline to tune

pipeline_to_tune = Pipeline(steps=[

    ("preprocessor", preprocessor),

    ("classifier", RandomForestClassifier(random_state=42))

])



# GridSearchCV: tries all combinations, evaluates each with 5-fold CV

grid_search = GridSearchCV(

    pipeline_to_tune,

    param_grid,

    cv=5,

    scoring="accuracy",

    n_jobs=-1,        # use all CPU cores — faster

    verbose=1         # prints progress so you know it's running

)



grid_search.fit(X_train, y_train)



# Results

print("=== Hyperparameter Tuning Results ===\n")

print(f"Best Parameters : {grid_search.best_params_}")

print(f"Best CV Score   : {grid_search.best_score_:.3f}")



# Evaluate best model on held-out test set

y_pred_tuned = grid_search.best_estimator_.predict(X_test)



print(f"\n=== Final Evaluation on Test Set ===\n")

print(f"Accuracy: {accuracy_score(y_test, y_pred_tuned):.3f}")

print(classification_report(y_test, y_pred_tuned))